#Retrieval Part

Installing imortant Libraries

In [3]:
#!pip install pypdf sentence-transformers faiss-cpu
#!pip install -U langchain-experimental langchain-community
#!pip install -U langchain-huggingface

Important Imports

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from google.colab import drive
from pypdf import PdfReader
import faiss
import os
import re

/tmp/ipykernel_885/3131809024.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [5]:
drive.mount('/content/drive')

Mounted at /content/drive


Getting a pdf Document

In [6]:
pdf_path='/content/drive/MyDrive/ACT-1975.pdf'
pdf_file=os.path.exists(pdf_path)
print(pdf_file ) #Checking if the path Exist

True


Extracting text Page wise

In [7]:

reader=PdfReader(pdf_path)

text=''
pages=[]

for page_number,page in enumerate(reader.pages,start=1):
  text=page.extract_text()
  pages.append({
      "page_number":page_number,
      "text":text
      })






Number of Pages

In [8]:
len(reader.pages)

12

Evaluating/looking at some pages/
Pages need to be cleaned





In [9]:
pages[2]['text'][:800]

' \nPage 3 of 12 \n \nTHE ABANDONED PROPERTIES (MANAGEMENT) ACT, 1975 \nACT No. XX OF 1975 \n[12th February, 1975]  \nAn Act provide for the 1[* * *] management of certain properties \n WHEREAS it is expedient to provide for the 1[* * *] management of certain properties, and \nfor matters connected therewith ; \n AND WHEREAS the Proclamation of Emergency referred to in Article 280 of the Constitution \nis still in force; \n It is hereby enacted as follows:— \n 1. Short title, extent and commencement .—(1) This act may be called the Abandoned \nProperties (1[* * *] Management) Act, 1975. \n (2) It extends to the whole of Pakistan. \n (3) It shall come into force at once. \n 2.  Definitions.—In this Act, unless there is anything repugnant in the subject or context,— \n(a) “abandoned property” means any proper'

length of characters in each page

In [10]:
for i in range(len(reader.pages)):
  print(f'Page {i+1}: {len(pages[i]['text'])} characters')

Page 1: 1109 characters
Page 2: 331 characters
Page 3: 1838 characters
Page 4: 2693 characters
Page 5: 3257 characters
Page 6: 3321 characters
Page 7: 2730 characters
Page 8: 2578 characters
Page 9: 2578 characters
Page 10: 2520 characters
Page 11: 2463 characters
Page 12: 583 characters


Cleaning the extracted text and combining the pages

In [11]:
def clean_text(text):

    # Remove page markers
    text = re.sub(r'Page \d+ of \d+', '', text)

    # Replace multiple spaces/tabs with one space
    text = re.sub(r'[ \t]+', ' ', text)

    # Fix escaped section numbers
    text = text.replace(r'\.', '.')

    # Remove excessive blank lines
    text = re.sub(r'\n\s*\n+', '\n\n', text)

    return text.strip()

In [12]:
cleaned_pages=[]
for page in pages:
  cleaned_pages.append({
      'Pages':page['page_number'],
      'text':clean_text(page['text'])
  })

Text is cleaned and has a better structure than raw text that we Extracted

In [13]:
cleaned_pages[0]['text'][:800]

'THE ABANDONED PROPERTIES (MANAGEMENT) ACT, 1975 \n\n CONTENTS \n1. Short title, extent and commencement \n2. Definitions \n3. Vesting of abandoned property in Government \n4. Board of Trustees \n5. Appointment of Administrator and Deputy Administrators \n6. Holding of abandoned property and its surrender, etc. \n7. Power of Administrator to take possession of abandoned property \n8. Payment to Administrator \n9. Recovery of damages for unauthorised possession \n10. Exemption from legal process \n11. Publication of list of abandoned property \n12. Prohibition of transfers of property generally \n13. Confirmation of transfers by specified persons \n14. Claims by interested persons \n15. Appeal and revision \n16. Powers and duties of the Administrator generally \n17. Evaluation of abandoned property \n18. Expend'

In [14]:
document=""
for c_page in cleaned_pages:
  if c_page['Pages']>2:
    document += f"\n[PAGE {c_page['Pages']}]\n"
    document+=c_page['text']

In [16]:
#print(document)

Finding Sections

In [17]:
section_pattern = re.compile(
    r'(?m)^\s*(\d{1,2})\.\s+(.+?)(?:\s*[.—-])(?:\([0-9]+\)|[A-Z])'
)

In [18]:
matches = section_pattern.finditer(document)

for match in matches:
    print(match.group(1), "→", match.group(2))

1 → Short title, extent and commencement .
2 → Definitions.
3 → Vesting of abandoned property in Government.
4 → Board of Trustees.
5 → Appointment of Administrator and Deputy Administrators .
6 → Holding of abandoned property and its surrender, etc.
7 → Power of Administrator to take possession of abandoned property.
8 → Payment to Administrator .
9 → Recovery of dama ges for unauthorised possession .
10 → Exemption from legal process .
11 → Publication of lis t of abandoned property.
12 → Prohibition of transfers of property generally .
13 → Confirmation of transfers by specified persons.
14 → Claims by interested persons .
15 → Appeal and revision.
16 → Powers and duties of the Administrator generally .
17 → Evaluation of abandoned property.
18 → Expenditure by A dministrator how to be recouped .
19 → Maintenance of accounts by Ad ministrator.
20 → Powers of Board and Administrator when holding an inquiry, etc.
21 → Recovery of arrears.
22 → Penalty and procedure.
23 → Bar of jurisd

In [19]:
section_pattern = re.compile(
    r'(?m)^\s*(\d{1,2})\.\s+([A-Z][^\n]+)'
)

matches = list(section_pattern.finditer(document))

print("Number of matches:", len(matches))

found_sections = [int(m.group(1)) for m in matches]

print("Sections found:", found_sections)
print("Number of section numbers:", len(found_sections))
print("Unique sections:", len(set(found_sections)))

Number of matches: 30
Sections found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
Number of section numbers: 30
Unique sections: 30


Dividing Document based on sections

In [20]:
sections = []

for i, match in enumerate(matches):

    start = match.start() #Gives the index position of where the section starts in doc

    if i + 1 < len(matches):
        end = matches[i + 1].start()
    else:
        end = len(document)

    section_text = document[start:end].strip()

    section_number = match.group(1)

    sections.append({
        "section": section_number,
        "text": section_text
    })

In [21]:
print("Number of sections:", len(sections))

Number of sections: 30


In [22]:
print(sections[29]["text"])

30. Power to make rules .—(1) The Federal Government may, by notification in the official 
Gazette, make such rules as appear to it to be necessary for carrying out the purposes of this Act. 
 (2) In particular, and without prejudice to the generality of the foregoing power, such rules 
may provide for all or any of the following matters, namely:— 
(a) the terms and conditions of service of the Administrato r and Deputy 
Administrators ; 
(b) the functions to be performed by the Administrator and the Deputy 
Administrators ; 
(c) the manner in which entry or search under clause (b) or clause (e) of sub­section 
(2) of section 16 may be made or possession of any abandoned property may be 
taken by the Administrator ; 
(d) the manner in which inquiries under this Act may be held ; 
(e) the time within which application for confirmation under section 13 or 
preferring claim under section 14 may be made ; 
(f) the income­tax authority who may issue a no­objection certificate under section 

Sections with more than 1000 Characters

In [23]:
print('Sections with more than 1000 characters')
for section in sections:
    if len(section['text'])>1000:
      print('SECTION:',section['section'])

Sections with more than 1000 characters
SECTION: 2
SECTION: 4
SECTION: 6
SECTION: 13
SECTION: 15
SECTION: 16
SECTION: 20
SECTION: 30


Extract Title of Each Section

In [24]:
for section in sections:

    first_line = section["text"].split("\n")[0].strip()

    # Remove section number
    title = re.sub(
        rf'^\s*{section["section"]}\.\s*',
        '',
        first_line
    )

    # Remove everything after the legal dash
    title = re.split(r'[.—]', title)[0].strip()

    section["title"] = title

In [27]:
for section in sections[-6:]:

    print(
        section["section"],
        "->",
        section["title"]
    )

25 -> Delegation of powers
26 -> Officers and servants
27 -> Act to override other laws
28 -> Power to exempt
29 -> Power of Federal Government to take action with regard to abandoned property
30 -> Power to make rules


In [28]:
def get_page_number(document, position):

    text_before = document[:position]

    pages_found = re.findall(
        r'\[PAGE (\d+)\]',
        text_before
    )

    if pages_found:
        return int(pages_found[-1])

    return None

In [29]:
for i, match in enumerate(matches):

    sections[i]["page"] = get_page_number(
        document,
        match.start()
    )

In [30]:
for section in sections[:10]:

    print(
        "Section:", section["section"],
        "| Page:", section["page"],
        "| Title:", section["title"]
    )

Section: 1 | Page: 3 | Title: Short title, extent and commencement
Section: 2 | Page: 3 | Title: Definitions
Section: 3 | Page: 4 | Title: Vesting of abandoned property in Government
Section: 4 | Page: 4 | Title: Board of Trustees
Section: 5 | Page: 4 | Title: Appointment of Administrator and Deputy Administrators
Section: 6 | Page: 5 | Title: Holding of abandoned property and its surrender, etc
Section: 7 | Page: 5 | Title: Power of Administrator to take possession of abandoned property
Section: 8 | Page: 5 | Title: Payment to Administrator
Section: 9 | Page: 5 | Title: Recovery of dama ges for unauthorised possession
Section: 10 | Page: 6 | Title: Exemption from legal process


##Selecting a Chunking Strategy

Recusrive Chunking

In [31]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)


Creating Chunks

In [32]:
chunks = []

for section in sections:

    text = section["text"]

    # Small section → keep as one chunk
    if len(text) <= 1000:

        section_chunks = [text]

    # Large section → recursively split
    else:

        section_chunks = splitter.split_text(text)

    for chunk in section_chunks:

        chunks.append({
            "text": chunk,
            "section": section["section"],
            "title": section["title"]
        })

In [33]:
print("Total chunks:", len(chunks))

Total chunks: 43


In [34]:
for i, chunk in enumerate(chunks[:10]):

    print("=" * 70)
    print("Chunk:", i)
    print("Section:", chunk["section"])
    print("Title:", chunk["title"])
    print("Length:", len(chunk["text"]))
    print(chunk["text"])

Chunk: 0
Section: 1
Title: Short title, extent and commencement
Length: 210
1. Short title, extent and commencement .—(1) This act may be called the Abandoned 
Properties (1[* * *] Management) Act, 1975. 
 (2) It extends to the whole of Pakistan. 
 (3) It shall come into force at once.
Chunk: 1
Section: 2
Title: Definitions
Length: 974
2. Definitions.—In this Act, unless there is anything repugnant in the subject or context,— 
(a) “abandoned property” means any property, movable or immovable (including 
share in industrial units and firms, investments, deposits, policies of insurance 
and all other interests and rights in or to or arising out of any such property), 
belonging to a specified person and includes any such property sold or 
transferred to, or placed unde r the supervision or control of, any other person 
on or after the sixteenth day of December , 1971, but does not include any 
ornaments or wearing apparel or any cooking vessels or other house-hold effects 
in the immedia

Creating MetaDeta

In [35]:
for i, chunk in enumerate(chunks):

    chunk["metadata"] = {
        "source": "Abandoned Properties (Management) Act, 1975",
        "section": chunk["section"],
        "title": chunk["title"],
        "chunk_id": i
    }

In [37]:
chunks = []

for section in sections:

    text = section["text"]

    if len(text) <= 1000:
        section_chunks = [text]
    else:
        section_chunks = splitter.split_text(text)

    for chunk_text in section_chunks:

        chunks.append({
            "text": chunk_text,

            "metadata": {
                "source": "Abandoned Properties (Management) Act, 1975",
                "page": section["page"],
                "section": section["section"],
                "title": section["title"]
            }
        })

In [41]:
chunks[0]

{'text': '1. Short title, extent and commencement .—(1) This act may be called the Abandoned \nProperties (1[* * *] Management) Act, 1975. \n (2) It extends to the whole of Pakistan. \n (3) It shall come into force at once.',
 'metadata': {'source': 'Abandoned Properties (Management) Act, 1975',
  'page': 3,
  'section': '1',
  'title': 'Short title, extent and commencement'}}

Embedding

In [39]:
embedding=SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [42]:
chunk_texts = [chunk["text"] for chunk in chunks]

print("Number of texts:", len(chunk_texts))

Number of texts: 43


In [45]:
embeddings=embedding.encode(
    chunk_texts,
    convert_to_numpy=True
)

In [46]:
embeddings.shape

(43, 384)

Create the FAISS

In [47]:
dimensions=embeddings.shape[1]

In [48]:
index=faiss.IndexFlatL2(dimensions)

In [49]:
index.add(embeddings)

In [50]:
print("Vectors Stored: ",index.ntotal)

Vectors Stored:  43


lets suppose a user comes

In [52]:
questions='What happens if a person refuses to surrender abandoned property?'
question_embedding=embedding.encode( #Embedd user Question
    [questions],
    convert_to_numpy=True
)


In [53]:
question_embedding.shape

(1, 384)

In [54]:
dist,indices=index.search(
    question_embedding,
    k=2
)

In [55]:
indices

array([[10, 25]])

In [59]:
def get_chunk(indices):
  final_chunks=[]
  for i in indices[0]:
    final_chunks.append(chunks[i])
  return final_chunks


In [79]:
retreived_chunks=get_chunk(indices)
retreived_chunks

[{'text': '7. Power of Administrator to take possession of abandoned property.—If any person who \nhas purchased any abandoned property the sale of which has not been confirmed by the Administrator \nunder section 13, or who is in possession, supervision or management of any abandoned property or \nproperty which he knows or has reason to believe to be abandoned propert y does not surrender such \nproperty to the Administrator or the person authorised by him in this behalf, then, without prejudice to \nany other action or penalty to which such person may otherwise be liable, the Administrator may use \nsuch force as is necessary for taking possession of such property and may for this purpose, after giving \nreasonable warning and facility to any woman not appearing in public to withdraw, remove or break \nopen any lock, bolt or door, or do any other act necessary for the said purpose.',
  'metadata': {'source': 'Abandoned Properties (Management) Act, 1975',
   'page': 5,
   'section': 

In [66]:
context = "\n\n".join(
    [chunk["text"] for chunk in retreived_chunks]
)

In [67]:
context

'7. Power of Administrator to take possession of abandoned property.—If any person who \nhas purchased any abandoned property the sale of which has not been confirmed by the Administrator \nunder section 13, or who is in possession, supervision or management of any abandoned property or \nproperty which he knows or has reason to believe to be abandoned propert y does not surrender such \nproperty to the Administrator or the person authorised by him in this behalf, then, without prejudice to \nany other action or penalty to which such person may otherwise be liable, the Administrator may use \nsuch force as is necessary for taking possession of such property and may for this purpose, after giving \nreasonable warning and facility to any woman not appearing in public to withdraw, remove or break \nopen any lock, bolt or door, or do any other act necessary for the said purpose.\n\n17. Evaluation of abandoned property.—(1) The Administrator may determine the value of \nany property of whic

#Generation Part

In [68]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name='Qwen/Qwen2.5-1.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='auto'
                                           )

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Wtring a prompt for the LLM

In [69]:
prompt = f"""
You are a helpful assistant answering questions using the provided context.

Answer the question using ONLY the information in the context.

If the answer cannot be found in the context, say:
"I don't have enough information in the provided documents."

Context:
{context}

Question:
{questions}

Answer:
"""

In [70]:
input=tokenizer(prompt,return_tensors='pt').to(model.device)

In [71]:
with torch.no_grad():
  outputs=model.generate(
    **input,
    max_new_tokens=512,
    temperature=0.1
  )

Number of tokens in the OG prompt

In [72]:
input_len=input['input_ids'].shape[1]
input_len

395

In [73]:
generated_info=tokenizer.decode(
    outputs[0][input_len:],
    skip_special_tokens=True
)


#Final Results

In [83]:
def result(questions,generated_info,retreived_chunks):
  print(f'Question: {questions}\n')
  print(f'Answer: {generated_info}')

  print('---------------------------')
  print('Citations from the document')

  print('Total sources found: ',len(retreived_chunks))
  for chunk in retreived_chunks:

    print(f'Page: {chunk["metadata"]["page"]}')
    print(f'Section: {chunk["metadata"]["section"]}')
    print(f'Title: {chunk["metadata"]["title"]}')

    print('-'*70)


In [84]:
result(questions,generated_info,retreived_chunks)

Question: What happens if a person refuses to surrender abandoned property?

Answer: The Administrator can use force to take possession of the property. They must give reasonable warning before doing so. If no one appears to withdraw, they can remove or break open locks, bolts, or doors. This action is authorized by law.
---------------------------
Citations from the document
Total sources found:  2
Page: 5
Section: 7
Title: Power of Administrator to take possession of abandoned property
----------------------------------------------------------------------
Page: 8
Section: 17
Title: Evaluation of abandoned property
----------------------------------------------------------------------
